In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/10000
1


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  28.72642317643522
RUN  2 , total integrated cost =  22.631388036519255
RUN  3 , total integrated cost =  13.181620834772184
RUN  4 , total integrated cost =  12.478884561969194
RUN  5 , total integrated cost =  11.644913316985487
RUN  6 , total integrated cost =  10.861791666681848
RUN  7 , total integrated cost =  9.882623172932007
RUN  8 , total integrated cost =  8.305192000932433
RUN  9 , total integrated cost =  7.903841543712083
RUN  10 , total integrated cost =  7.071803320814606
RUN  11 , total integrated cost =  7.062200408851709
RUN  12 , total integrated cost =  7.0600891072469585
RUN  13 , total integrated cost =  7.041463954178724
RUN  14 , total integrated cost =  7.033327523357311
RUN  15 , total integrated cost =  7.031696048694762
RUN  1

ERROR:root:Problem in initial value trasfer


RUN  50 , total integrated cost =  6.733750503735941
State only changes marginally.
Control only changes marginally.
RUN  57 , total integrated cost =  6.733750503665019
Improved over  57  iterations in  13.154527304694057  seconds by  99.86789547523054  percent.
Problem in initial value trasfer:  Vmean_exc -67.77847799261714 -67.78255789332957
weight =  7569.763425932384
set cost params:  1.0 0.0 7569.763425932384
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5049.491760907217
Gradient descend method:  None
RUN  1 , total integrated cost =  4758.41131512083
RUN  2 , total integrated cost =  4758.41131512082
RUN  3 , total integrated cost =  4758.4113151208185
RUN  4 , total integrated cost =  4758.411315120817


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  4758.411315120816
RUN  6 , total integrated cost =  4758.411315120816
Control only changes marginally.
RUN  6 , total integrated cost =  4758.411315120816
Improved over  6  iterations in  0.3032498639076948  seconds by  5.764549375838655  percent.
Problem in initial value trasfer:  Vmean_exc -63.10648565278801 -63.153756443679356
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  95.57051029435236
RUN  2 , total integrated cost =  82.7948669170854
RUN  3 , total integrated cost =  63.85564794593739
RUN  4 , total integrated cost =  56.99491247265159
RUN  5 , total integrated cost =  49.819075703823174
RUN  6 , total integrated cost =  45.93053204119043
RUN  7 , total integrated cost =  42.4068008959097
RUN  8 , total integrated cost =  39.31304428429711
RUN  9 , total i

...

RUN  30 , total integrated cost =  20377.816159267426
Control only changes marginally.
RUN  33 , total integrated cost =  20377.816159264632
Improved over  33  iterations in  1.3428039718419313  seconds by  10.757265798075522  percent.
Problem in initial value trasfer:  Vmean_exc -57.10428211831827 -57.092415631502824
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  203.18583579668189
RUN  2 , total integrated cost =  125.03510087096298
RUN  3 , total integrated cost =  65.18571327706353
RUN  4 , total integrated cost =  62.22678372254721
RUN  5 , total integrated cost =  61.29857467042241
RUN  6 , total integrated cost =  61.23391008005912
RUN  7 , total integrated cost =  60.71136999750381
RUN  8 , total integrated cost =  60.29100557992929
RUN  9 , total integrated cost =  60.279241370333835
RUN  1

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  47 , total integrated cost =  57.826484730910636
Improved over  47  iterations in  1.8360515404492617  seconds by  99.82629499750566  percent.
Problem in initial value trasfer:  Vmean_exc -64.38048122361944 -64.39658541467054
weight =  5756.886593018651
set cost params:  1.0 0.0 5756.886593018651
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32074.904603659263
Gradient descend method:  None
RUN  1 , total integrated cost =  28946.07385428381
RUN  2 , total integrated cost =  28773.66474595628
RUN  3 , total integrated cost =  28633.514841705717
RUN  4 , total integrated cost =  28594.35131336651
RUN  5 , total integrated cost =  28562.402784824153
RUN  6 , total integrated cost =  28443.81698875222
RUN  7 , total integrated cost =  28410.65266063931
RUN  8 , total integrated cost =  28410.2964656747
RUN  9 , total integrated cost =  28403.035942739512
RUN  10 , total integrated cost =  28398.197449214473
RUN  11 , total in

ERROR:root:Problem in initial value trasfer


RUN  70 , total integrated cost =  27080.18926603866
Control only changes marginally.
RUN  72 , total integrated cost =  27077.474241965643
Improved over  72  iterations in  2.80540837906301  seconds by  15.58049953209678  percent.
Problem in initial value trasfer:  Vmean_exc -56.65946466763812 -56.66307957878911


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
------- 

ERROR:root:Problem in initial value trasfer


RUN  70 , total integrated cost =  54.83806097877562
Control only changes marginally.
RUN  71 , total integrated cost =  54.83806097877562
Improved over  71  iterations in  1.5968349371105433  seconds by  99.8168297684153  percent.
Problem in initial value trasfer:  Vmean_exc -62.79310871589771 -62.795498432260246
weight =  5570.297059930752
set cost params:  1.0 0.0 5570.297059930752
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  29570.8293560735
Gradient descend method:  None
RUN  1 , total integrated cost =  26716.117283532003
RUN  2 , total integrated cost =  26657.76433890952
RUN  3 , total integrated cost =  26609.69702566544
RUN  4 , total integrated cost =  26510.294075088532
RUN  5 , total integrated cost =  26425.940405926518
RUN  6 , total integrated cost =  26309.38285993457
RUN  7 , total integrated cost =  26237.434977652214
RUN  8 , total integrated cost =  26136.40099925178
RUN  9 , total integrated cost =  26125.579509814423
RUN  10 , total int

ERROR:root:Problem in initial value trasfer


RUN  40 , total integrated cost =  24943.216976617234
Control only changes marginally.
RUN  44 , total integrated cost =  24774.505583370636
Improved over  44  iterations in  0.9994845017790794  seconds by  16.21978103809171  percent.
Problem in initial value trasfer:  Vmean_exc -56.653811459382375 -56.65721326985208
-------  40 0.5250000000000001 0.5500000000000003
[0, 5, 10, 15, 20, 25, 30] []
closest index  30
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  25457.689324367344
Gradient descend method:  None
RUN  1 , total integrated cost =  166.5287394458541
RUN  2 , total integrated cost =  130.39606372607156
RUN  3 , total integrated cost =  57.374392999430306
RUN  4 , total integrated cost =  56.60732197478359
RUN  5 , total integrated cost =  55.4392555082049
RUN  6 , total integrated cost =  54.81625835359673
RUN  7 , total integrated cost =  52.43767411258395
RUN  8 , total integrated cost =  52.050764

...

RUN  40 , total integrated cost =  31466.395565754527
Control only changes marginally.
RUN  44 , total integrated cost =  31466.395565754483
Improved over  44  iterations in  0.9734572879970074  seconds by  15.534467542109994  percent.
Problem in initial value trasfer:  Vmean_exc -56.66017804779251 -56.664403757796556
-------  135 0.5250000000000001 0.8750000000000006
-------  140 0.4500000000000001 0.9000000000000006
-------  145 0.5750000000000002 0.9000000000000006
[0, 5, 10, 15, 20, 25, 30, 50, 55, 70, 85, 100, 110, 115, 125, 140, 65, 45, 80, 95, 135, 120] [125, 110, 140, 115, 95, 100, 135, 85, 80, 120, 70, 65, 55, 50, 45, 30, 25, 20, 15, 10, 5]
closest index  0
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33431.735644455905
Gradient descend method:  None
RUN  1 , total integrated cost =  202.25961629951482
RUN  2 , total integrated cost =  124.68568273431575
RUN  3 , total integrated cost =  65.2320804

ERROR:root:Problem in initial value trasfer


RUN  40 , total integrated cost =  57.755434889967766
Control only changes marginally.
RUN  42 , total integrated cost =  57.75543488996776
Improved over  42  iterations in  0.9478415753692389  seconds by  99.8272436839529  percent.
Problem in initial value trasfer:  Vmean_exc -64.24634859900702 -64.26293734354955
weight =  5763.96862568864
set cost params:  1.0 0.0 5763.96862568864
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32110.96249880973
Gradient descend method:  None
RUN  1 , total integrated cost =  29046.39261575453
RUN  2 , total integrated cost =  29012.029969078947
RUN  3 , total integrated cost =  28985.540958532332
RUN  4 , total integrated cost =  28953.648793304965
RUN  5 , total integrated cost =  28928.26765454123
RUN  6 , total integrated cost =  28897.382607864536
RUN  7 , total integrated cost =  28870.023651243468
RUN  8 , total integrated cost =  28832.6430561955
RUN  9 , total integrated cost =  28797.91221360409
RUN  10 , total integ

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  68 , total integrated cost =  27103.827247270896
Improved over  68  iterations in  1.503211384639144  seconds by  15.593226929041563  percent.
Problem in initial value trasfer:  Vmean_exc -56.65114605748814 -56.654641715527816
------------------------------------------------------------
-------------------- 22
------------------------------------------------------------
found solution:  [0, 5, 10, 15, 20, 25, 30, 50, 55, 70, 85, 100, 110, 115, 125, 140, 65, 45, 80, 95, 135, 120]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
closest index  -1
set cost params:  1.0 0.0 10.0
all options 

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  8107.857254577991
set cost params:  1.0 0.0 8107.857254577991
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5092.1269366155175
Gradient descend method:  None
RUN  1 , total integrated cost =  5091.991494000924
RUN  2 , total integrated cost =  5091.989614351656
RUN  3 , total integrated cost =  5091.989524029981
RUN  4 , total integrated cost =  5091

ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  5091.989520326497
RUN  8 , total integrated cost =  5091.989520326496
RUN  9 , total integrated cost =  5091.989520326495
RUN  10 , total integrated cost =  5091.989520326495
Control only changes marginally.
RUN  10 , total integrated cost =  5091.989520326495
Improved over  10  iterations in  0.2980985715985298  seconds by  0.002698602975385711  percent.
Problem in initial value trasfer:  Vmean_exc -62.90359738789991 -62.95034738643959
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  5776.497294854673
set cost params:  1.0 0.0 5776.497294854673
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13004.29194832725
Gradient descend method:  None
RUN  1 , total integrated cost =  13004.006274438088
RUN  2 , total integrated cost =  13004.000732113855
RUN  3 , total integrated cost =  13003.999725998707
RUN  4 , total integrated cost =  13003

ERROR:root:Problem in initial value trasfer


RUN  14 , total integrated cost =  13003.999219461344
RUN  15 , total integrated cost =  13003.99921946134
RUN  16 , total integrated cost =  13003.999219461339
RUN  17 , total integrated cost =  13003.999219461339
Control only changes marginally.
RUN  17 , total integrated cost =  13003.999219461339
Improved over  17  iterations in  0.42475626803934574  seconds by  0.0022510173339327366  percent.
Problem in initial value trasfer:  Vmean_exc -58.73862513572914 -58.74652666128539
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  6457.054231601654
set cost params:  1.0 0.0 6457.054231601654
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8225.252234172687
Gradient descend method:  None
RUN  1 , total integrated cost =  8225.252234172685


...

RUN  4 , total integrated cost =  39155.80570454728
Control only changes marginally.
RUN  4 , total integrated cost =  39155.80570454728
Improved over  4  iterations in  0.19828899390995502  seconds by  0.004589541169309541  percent.
Problem in initial value trasfer:  Vmean_exc -56.70273089161024 -56.7024818781781
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  105 0.5750000000000002 0.7750000000000005
no convergence
weight =  8349.79660135455
set cost params:  1.0 0.0 8349.79660135455
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33737.99916433082
Gradient descend method:  None
RUN  1 , total integrated cost =  33736.84217944574
RUN  2 , total integrated cost =  33736.84217944572
RUN  3 , total integrated cost =  33736.84217944571


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  33736.84217944571
Control only changes marginally.
RUN  4 , total integrated cost =  33736.84217944571
Improved over  4  iterations in  0.18483829125761986  seconds by  0.003429322762954712  percent.
Problem in initial value trasfer:  Vmean_exc -56.70424807502851 -56.70423127584801
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  130 0.6000000000000003 0.8500000000000005
no convergence
weight =  8676.285876603524
set cost params:  1.0 0.0 8676.285876603524
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  38546.97530946541
Gradient descend method:  None
RUN  1 , total integrated cost =  38545.471780765976
RUN  2 , total integrated cost =  38545.46891789442
RUN  3 , total integrated cost =  3854

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  38545.468917894395
Control only changes marginally.
RUN  5 , total integrated cost =  38545.468917894395
Improved over  5  iterations in  0.19986964389681816  seconds by  0.003907937156981234  percent.
Problem in initial value trasfer:  Vmean_exc -56.70300230196636 -56.70279408273245
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
-------  145 0.5750000000000002 0.9000000000000006
no convergence
weight =  8310.954066828646
set cost params:  1.0 0.0 8310.954066828646
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33139.7261148264
Gradient descend method:  None
RUN  1 , total integrated cost =  33138.20398914632
RUN  2 , total integrated cost =  33138.20213501358
RUN  3 , total integrated cost =  33138.20213501357
RUN  4 , total integrated cost =  33138.20213501357
Control only changes marginally.
RUN  4 , total integrated cost =  33138.20

In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [False, False], [True, True], [True, True], [False, False], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [False, False]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  75.00637025918333
Gradient descend method:  None
RUN  1 , total integrated cost =  6.73359680831561
RUN  2 , total integrated cost =  6.7335968083156015
RUN  3 , total integrated cost =  6.733596808315598


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  6.733596808315598
Control only changes marginally.
RUN  4 , total integrated cost =  6.733596808315598
Improved over  4  iterations in  0.5305809136480093  seconds by  91.02263343093692  percent.
Problem in initial value trasfer:  Vmean_exc -67.7777040658539 -67.78179757807003
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  66.75416623561193
Gradient descend method:  HS
RUN  1 , total integrated cost =  65.92980704445915
RUN  2 , total integrated cost =  65.63025703809481
RUN  3 , total integrated cost =  65.55103414378107


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  65.55103414378104
RUN  5 , total integrated cost =  65.55103414378104
Control only changes marginally.
RUN  5 , total integrated cost =  65.55103414378104
Improved over  5  iterations in  0.654156357049942  seconds by  1.8023325878783112  percent.
Problem in initial value trasfer:  Vmean_exc -67.55928133874698 -67.56706845684083
weight =  7775.063177003766
set cost params:  1.0 0.0 7775.063177003766
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5052.204471440633
Gradient descend method:  None
RUN  1 , total integrated cost =  4891.88151695393
RUN  2 , total integrated cost =  4891.863923500605
RUN  3 , total integrated cost =  4891.863908982581
RUN  4 , total integrated cost =  4891.8639089825765
RUN  5 , total integrated cost =  4891.86390898257
RUN  6 , total integrated cost =  4891.863908982569


...

Control only changes marginally.
RUN  54 , total integrated cost =  57.43850895084396
Improved over  54  iterations in  6.090573627501726  seconds by  99.79270352121542  percent.
Problem in initial value trasfer:  Vmean_exc -64.28925728373598 -64.30591252522434
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  557.0204669598551
Gradient descend method:  HS
RUN  1 , total integrated cost =  537.8377052760723
RUN  2 , total integrated cost =  535.3619719670779
RUN  3 , total integrated cost =  535.2445613255875
RUN  4 , total integrated cost =  535.1957228217951
RUN  5 , total integrated cost =  535.1957132918232
RUN  6 , total integrated cost =  535.195713291823
RUN  7 , total integrated cost =  535.1957132918227


ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  535.1957132918227
Control only changes marginally.
RUN  8 , total integrated cost =  535.1957132918227
Improved over  8  iterations in  1.026227070018649  seconds by  3.918124191584738  percent.
Problem in initial value trasfer:  Vmean_exc -61.38644765042689 -61.395354496121875
weight =  6219.164070844467
set cost params:  1.0 0.0 6219.164070844467
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32396.7265093326
Gradient descend method:  None
RUN  1 , total integrated cost =  30959.1975811248
RUN  2 , total integrated cost =  30949.363827730886
RUN  3 , total integrated cost =  30934.60696814434
RUN  4 , total integrated cost =  30923.1596381063
RUN  5 , total integrated cost =  30879.611592121302
RUN  6 , total integrated cost =  30844.27627306787
RUN  7 , total integrated cost =  30818.49967468055
RUN  8 , total integrated cost =  30800.79475175025
RUN  9 , total integrated cost =  30798.093931682222
RUN  10 , total integrated

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  57 , total integrated cost =  28857.84241328199
Improved over  57  iterations in  7.461401799693704  seconds by  10.923585427778193  percent.
Problem in initial value trasfer:  Vmean_exc -56.68049817559099 -56.683458703023405


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  8100.564390002482
set cost params:  1.0 0.0 8100.564390002482
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5094.336374563857
Gradient descend method:  None
RUN  1 , total integrated cost =  5094.291934966574
RUN  2 , total integrated cost =  5094.29130916049
RUN  3 , total integrated cost =  5094.291291594122
RUN  4 , total integrated cost =  5094.291291472553
RUN  5 , total integrated cost =  5094.291291472538
RUN  6 , total integ

ERROR:root:Problem in initial value trasfer


RUN  10 , total integrated cost =  5094.291291472529
RUN  11 , total integrated cost =  5094.291291472529
Control only changes marginally.
RUN  11 , total integrated cost =  5094.291291472529
Improved over  11  iterations in  1.4901281483471394  seconds by  0.0008849649495630274  percent.
Problem in initial value trasfer:  Vmean_exc -63.351286794897405 -63.39835801439233
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  5793.6983921281735
set cost params:  1.0 0.0 5793.6983921281735
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13011.304276138872
Gradient descend method:  None
RUN  1 , total integrated cost =  13011.241357356217
RUN  2 , total integrated cost =  13011.240566766244
RUN  3 , total integrated cost =  13011.240557493311
RUN  4 , total integrated cost =  13011.240557359071
RUN  5 , total integrated cost =  13011.240557359055
RUN  6 , total integrated cost =  13011.240557359053


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  13011.240557359053
Control only changes marginally.
RUN  7 , total integrated cost =  13011.240557359053
Improved over  7  iterations in  1.045276764780283  seconds by  0.0004897186205710113  percent.
Problem in initial value trasfer:  Vmean_exc -59.31909794203786 -59.33258265007376
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  6508.5549527452995
set cost params:  1.0 0.0 6508.5549527452995
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8229.660696968649
Gradient descend method:  None
RUN  1 , total integrated cost =  8229.656695113075
RUN  2 , total integrated cost =  8229.656186500539
RUN  3 , total integrated cost =  8229.656156822728
RUN  4 , total integrated cost =  8229.656147860449
RUN  5 , total integrated cost =  8229.656142401047
RUN  6 , total integrated cost =  8229.656138536333
RUN  7 , total integrated cost =  8229.656134205587
RUN  8 , total integrated cost =  8229.656133940254
RUN  9

...

RUN  5 , total integrated cost =  34490.915679539736
Control only changes marginally.
RUN  5 , total integrated cost =  34490.915679539736
Improved over  5  iterations in  0.7941495161503553  seconds by  9.305027504069585e-08  percent.
Problem in initial value trasfer:  Vmean_exc -56.703461535844035 -56.70343044833701
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 99
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [T

ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  34490.94899189508
Control only changes marginally.
RUN  5 , total integrated cost =  34490.94899189508
Improved over  5  iterations in  0.762266805395484  seconds by  1.0295832453266485e-07  percent.
Problem in initial value trasfer:  Vmean_exc -56.703460603100204 -56.703429603240544
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 100
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [T

ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  34490.98104676506
Control only changes marginally.
RUN  3 , total integrated cost =  34490.98104676506
Improved over  3  iterations in  0.50088956579566  seconds by  2.112488459715678e-07  percent.
Problem in initial value trasfer:  Vmean_exc -56.70345947615777 -56.703428582210144
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 101


In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  58.758253431683606
Gradient descend method:  None
RUN  1 , total integrated cost =  0.6804247826762749
RUN  2 , total integrated cost =  0.6803140598244999
RUN  3 , total integrated cost =  0.6803140187048833
RUN  4 , total integrated cost =  0.6803140187048832
RUN  5 , total integrated cost =  0.680314018704883
State only changes marginally.


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  0.680314018704883
Control only changes marginally.
RUN  6 , total integrated cost =  0.680314018704883
Improved over  6  iterations in  0.21978462487459183  seconds by  98.84218134649652  percent.
Problem in initial value trasfer:  Vmean_exc -67.9000553631729 -67.90305637300891
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  109.35335465814576
Gradient descend method:  None
RUN  1 , total integrated cost =  2.481912862279189
RUN  2 , total integrated cost =  2.481804080408465
RUN  3 , total integrated cost =  2.481783553945108
RUN  4 , total integrated cost =  2.4817775365387846
RUN  5 , total integrated cost =  2.4817559866661423
RUN  6 , total integrated cost =  2.4813971306277782
RUN  7 , total integrated cost =  2.478872465874623
RUN  8 , total integrated cost =  2.4785141989266806
RUN  9 , total integrated cost =  2.4785042451702055
RUN  10 , to

ERROR:root:Problem in initial value trasfer


RUN  30 , total integrated cost =  2.4456839006466238
Control only changes marginally.
RUN  33 , total integrated cost =  2.4456839006466207
Improved over  33  iterations in  0.743130749091506  seconds by  97.7635035447315  percent.
Problem in initial value trasfer:  Vmean_exc -67.35729459366702 -67.3651397029292
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  46.23167761968753
Gradient descend method:  None
RUN  1 , total integrated cost =  1.319102107377302
RUN  2 , total integrated cost =  1.3190556568255718
RUN  3 , total integrated cost =  1.3190535342785763
RUN  4 , total integrated cost =  1.31905331450008
RUN  5 , total integrated cost =  1.3190532123982401
RUN  6 , total integrated cost =  1.3190531509409713
RUN  7 , total integrated cost =  1.3190530845950672
RUN  8 , total integrated cost =  1.3190530827761189
RUN  9 , total integrated cost =  1.319053082768268
RUN  10 , t

...

RUN  2 , total integrated cost =  0.7221963789230671
State only changes marginally.
RUN  3 , total integrated cost =  0.7221963789230671
Control only changes marginally.
RUN  3 , total integrated cost =  0.7221963789230671
Improved over  3  iterations in  0.13505411334335804  seconds by  98.56282253282487  percent.
Problem in initial value trasfer:  Vmean_exc -75.36229361772637 -75.39857595664141
-------  125 0.47500000000000014 0.8500000000000005
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  116.69920105260833
Gradient descend method:  None
RUN  1 , total integrated cost =  2.6445658685806164
RUN  2 , total integrated cost =  2.6443055897031034
RUN  3 , total integrated cost =  2.64429441521559
RUN  4 , total integrated cost =  2.6442928814932607
RUN  5 , total integrated cost =  2.644291846384276
RUN  6 , total integrated cost =  2.6442913380994537
RUN  7 , total integrated cost =  2.644290883017997
RUN  8 , total integrated cos

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  47 , total integrated cost =  2.639336232345323
Improved over  47  iterations in  0.9886336605995893  seconds by  97.73834250060075  percent.
Problem in initial value trasfer:  Vmean_exc -71.95567233907298 -71.98591142864286
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  105.28502535908706
Gradient descend method:  None
RUN  1 , total integrated cost =  4.0885651520106965
RUN  2 , total integrated cost =  4.0885651520106965
Control only changes marginally.
RUN  2 , total integrated cost =  4.0885651520106965
Improved over  2  iterations in  0.0871782936155796  seconds by  96.11666983213789  percent.
Problem in initial value trasfer:  Vmean_exc -69.15255667957655 -69.17373069228422
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32041.6354973

ERROR:root:Problem in initial value trasfer


RUN  70 , total integrated cost =  5.965696851980219
Control only changes marginally.
RUN  74 , total integrated cost =  5.965695991036413
Improved over  74  iterations in  1.6268233601003885  seconds by  99.98138142482917  percent.
Problem in initial value trasfer:  Vmean_exc -66.02889677297671 -66.04133158017123


In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.680314018704883
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  0.680314018704883
Control only changes marginally.
RUN  1 , total integrated cost =  0.680314018704883
Improved over  1  iterations in  0.06280006468296051  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.9000553631729 -67.90305637300891
-------  15 0.4500000000000001 0.4500000000000002
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.4456839006466207
Gradient descend method:  None
RUN  1 , total integrated cost =  2.4456839006466207
Control only changes marginally.
RUN  1 , total integrated cost =  2.4456839006466207
Improved over  1  iterations in  0.06168348155915737  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.35729459366702 -67.3651397029292
-------  25 0.4250000000000001 0.5000000000000002
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.31905308167375

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  1.3190530816737556
Control only changes marginally.
RUN  1 , total integrated cost =  1.3190530816737556
Improved over  1  iterations in  0.062401825562119484  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -71.12727306935786 -71.14640593900388
-------  45 0.5000000000000002 0.5750000000000003
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  3.746336740195524
Gradient descend method:  None


...

RUN  1 , total integrated cost =  0.7221963789230671
Control only changes marginally.
RUN  1 , total integrated cost =  0.7221963789230671
Improved over  1  iterations in  0.061433663591742516  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -75.36229361772637 -75.39857595664141
-------  125 0.47500000000000014 0.8500000000000005
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.639336232345323
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  2.639336232345323
Control only changes marginally.
RUN  1 , total integrated cost =  2.639336232345323
Improved over  1  iterations in  0.0623735636472702  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -71.95567233907298 -71.98591142864286
-------  135 0.5250000000000001 0.8750000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4.0885651520106965
Gradient descend method:  None
RUN  1 , total integrated cost =  4.0885651520106965
Control only changes marginally.
RUN  1 , total integrated cost =  4.0885651520106965
Improved over  1  iterations in  0.06153598241508007  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -69.15255667957655 -69.17373069228422
-------  145 0.5750000000000002 0.9000000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5.96569599103

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5.965695991036413
Control only changes marginally.
RUN  1 , total integrated cost =  5.965695991036413
Improved over  1  iterations in  0.06354798935353756  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -66.02889677297671 -66.04133158017123
--------------- 12
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 0.

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
